# Per-Artifact Batch Execute

## What you'll learn

- Group artifacts into execution units to share setup work
- Control concurrent Modal calls independently of unit size
- Compare four concurrent calls with one call at a time

**Prerequisites:** [First Pipeline](../01-getting-started/01-first-pipeline.ipynb),
[Batching and Performance](01-batching-and-performance.ipynb),
[Running on Modal](../07-compute-backends/04-modal-execution.ipynb).
**Estimated time:** 10 minutes
**Modal account required:** Yes, with proxy-auth tokens configured.

Deploy the existing example endpoint before running these cells:

```bash
artisan modal deploy wait_tool
```

The deployment uses `WaitTool`'s current source overlay. If you also plan to
run the R2 tutorial, use its setup instructions so the shared deployment
retains the required secret and output policy.

In [ ]:
from __future__ import annotations

import time

from artisan.operations.examples import DataGenerator, WaitTool
from artisan.orchestration import PipelineManager, StepDisposition
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_pipeline, inspect_step

In [ ]:
env = tutorial_setup("batch_execute", clean=True)

## Group inputs, then dispatch their work

`artifacts_per_unit` controls how many artifacts share input loading,
materialization, postprocessing, and staging. `WaitTool` uses the default
per-artifact dispatch: each artifact produces one call to its deployed
Modal endpoint.

For a unit containing four artifacts:

```text
Load and materialize four inputs
    ├─ artifact 0 → endpoint call
    ├─ artifact 1 → endpoint call
    ├─ artifact 2 → endpoint call
    └─ artifact 3 → endpoint call
Collect the four results, then postprocess and stage them together
```

We'll keep the unit size fixed and change `max_concurrent_calls`, the
client's limit on simultaneous endpoint calls within each unit.

## Compare two concurrency limits

We'll generate eight datasets and run `WaitTool` twice. Each call waits
10 seconds and writes a marker file. Both runs use four artifacts per
unit and one local worker, so the two units execute in sequence.

The first run allows four simultaneous endpoint calls. The second allows
one. Both make eight calls and produce eight outputs; only the number of
calls in flight changes. Container startup and network delays affect the
measured times, so this comparison does not promise a fixed speedup.

Both measured steps set `skip_cache=True` so they execute the waits even
when you rerun a cell.

### Generate source data

In [ ]:
pipeline = PipelineManager.create(
    name="batch_execute_demo",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

generated = pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 8, "seed": 42},
)
print("Generated 8 source datasets")

### Allow four calls at once

`max_concurrent_calls=4` lets each unit's four calls overlap. Modal decides
how many containers are available. This client setting does not pre-warm
containers: `min_containers`, hardware, and other deployment settings take
effect when you deploy the endpoint, not when you override a pipeline step.

In [ ]:
t0 = time.perf_counter()
parallel = pipeline.run(
    operation=WaitTool,
    skip_cache=True,
    name="parallel",
    inputs={"dataset": output("generate", "datasets")},
    params={"seconds": 10},
    compute_provider={"active": "modal", "modal": {"max_concurrent_calls": 4}},
    batch_strategy={"artifacts_per_unit": 4, "max_workers": 1},
)
parallel_time = time.perf_counter() - t0
print(f"Up to four concurrent calls: {parallel_time:.1f}s")

### Send one call at a time

Keep the same operation and unit size, then set `max_concurrent_calls=1`.
The client waits for each call before sending the next. The eight waits
now take at least 80 seconds, plus transport and startup time. Sequential
calls may run on different containers.

In [ ]:
t0 = time.perf_counter()
sequential = pipeline.run(
    operation=WaitTool,
    skip_cache=True,
    name="sequential",
    inputs={"dataset": output("generate", "datasets")},
    params={"seconds": 10},
    compute_provider={"active": "modal", "modal": {"max_concurrent_calls": 1}},
    batch_strategy={"artifacts_per_unit": 4, "max_workers": 1},
)
sequential_time = time.perf_counter() - t0
print(f"One call at a time: {sequential_time:.1f}s")

In [ ]:
summary = pipeline.finalize()
assert summary["overall_success"], summary
assert parallel.disposition is StepDisposition.EXECUTED
assert sequential.disposition is StepDisposition.EXECUTED

for step in (generated, parallel, sequential):
    artifacts = inspect_step(
        env.delta_root,
        step.step_number,
        pipeline_run_id=pipeline.config.pipeline_run_id,
    )
    assert artifacts.height == 8, (step.step_name, artifacts)

print(f"Four concurrent calls: {parallel_time:.1f}s")
print(f"One call at a time:    {sequential_time:.1f}s")
print("Both runs produced all 8 marker artifacts.")
inspect_pipeline(env.delta_root, pipeline_run_id=pipeline.config.pipeline_run_id)

## When an operation needs the whole batch

`per_artifact_dispatch=False` passes a unit's prepared batch to one execute
call. This is useful locally when an operation can process several files
in one call. The bundled `SequentialSlowTransformer` demonstrates this
local Python contract; `SlowTransformer` processes each artifact separately.

Neither Python operation is a Modal command operation. The endpoint
protocol carries one file per input role, so disabling per-artifact
dispatch does not make a list of files portable to an endpoint. The
comparison above keeps per-artifact dispatch enabled and controls client
concurrency instead.

See [Configure Execution](../../how-to-guides/configuring-execution.md)
for the local batch and remote input contracts.

## Read the results

Compare the two measured times with the step overview above. Both steps
completed the same eight waits. Grouping four artifacts into each unit
shared setup and staging work; changing the client concurrency limit
changed how many of their endpoint calls could overlap.

For larger jobs, tune unit size for setup cost and cache granularity,
then set the call limit to suit the endpoint's capacity. More local workers
can each dispatch calls, so `max_concurrent_calls` is not a deployment-wide
limit.

## Summary

| Setting | What you observed |
|---------|-------------------|
| `artifacts_per_unit=4` | Four artifacts share one unit's setup and staging |
| `max_workers=1` | Units run one at a time |
| `max_concurrent_calls=4` | Up to four endpoint calls overlap within a unit |
| `max_concurrent_calls=1` | Each endpoint call finishes before the next begins |

The operation, input data, and output count stay the same in both runs.

## Next steps

- [Batching and Performance](01-batching-and-performance.ipynb) — `artifacts_per_unit` and overhead amortization
- [Compute Routing](../07-compute-backends/01-compute-routing.ipynb) — Switching between local and Modal compute
- [Running on Modal](../07-compute-backends/04-modal-execution.ipynb) — GPU selection, sandbox transport, debugging
- [Execution Flow](../../concepts/execution-flow.md) — How the framework dispatches and tracks work